In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import sklearn
print(sklearn.__version__)  # anota la versión

In [ ]:
# --- Carga del dataset Heart Disease (UCI) desde OpenML -------------
try:
    # Utiliza return_X_y para obtener las características y el objetivo por separado, en aras de la claridad y la solidez
    X, y = fetch_openml('heart-c', version=1, as_frame=True, return_X_y=True, parser='auto')
    df = pd.concat([X, y], axis=1)

    # La columna objetivo es «class»: «1» = sin enfermedad, «2» = con enfermedad
    # Asigna el valor «1» a 0 (sin enfermedad) y el valor «2» a 1 (con enfermedad)
    df['objetivo'] = (df['class'].astype(int) > 1).astype(int)
    df = df.drop(columns=['class'])
except Exception as e:
    print(f'Sin conexión a OpenML o error al procesar ({e}). Cargando desde respaldo...')
    # La URL original daba un error 404. He utilizado una alternativa habitual del repositorio de la UCI.
    # Este conjunto de datos concreto contiene valores perdidos, representados por «?», y no tiene encabezado.
    url = ('https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data')
    # Define las columnas según la documentación de la UCI para «processed.cleveland.data»
    cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
            'thalach','exang','oldpeak','slope','ca','thal','target_raw']
    df = pd.read_csv(url, header=None, names=cols, na_values='?')

    # En este conjunto de datos, los valores > 0 en «target_raw» indican una enfermedad cardíaca
    df['objetivo'] = (df['target_raw'].astype(int) > 0).astype(int)
    df = df.drop(columns=['target_raw'])

print(df.shape)
print(df['objetivo'].value_counts())


In [ ]:
# --- Exploración básica ----------------------------------------------
print(df.isnull().sum().sum())   # total de nulos
print(df.dtypes)                 # tipos de cada columna

# Separar features y target
X = df.drop(columns=['objetivo'])
y = df['objetivo']

# --- Split estratificado ---------------------------------------------
semilla = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=semilla, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# --- Árbol con profundidad controlada --------------------------------
arbol = DecisionTreeClassifier(max_depth=3, random_state=semilla)
arbol.fit(X_train, y_train)

acc_train = accuracy_score(y_train, arbol.predict(X_train))
acc_test  = accuracy_score(y_test,  arbol.predict(X_test))

print(f'Árbol(depth=3) | Train:{acc_train:.4f} Test:{acc_test:.4f}')


In [ ]:
# --- Visualizar el árbol ---------------------------------------------
fig, ax = plt.subplots(figsize=(16, 6))

plot_tree(
    arbol,
    feature_names=X_train.columns.tolist(),
    class_names=['Sin enfermedad', 'Con enfermedad'],
    filled=True,          # colorea nodos según clase dominante
    rounded=True,         # bordes redondeados
    fontsize=9,
    ax=ax,
)
ax.set_title('Árbol de decisión — Heart Disease (max_depth=3)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Comparar árboles con distintas profundidades -------------------
profundidades = [1, 2, 3, 5, 10, None]
resultados = []

for prof in profundidades:
    dt = DecisionTreeClassifier(max_depth=prof, random_state=semilla)
    dt.fit(X_train, y_train)
    acc_tr = accuracy_score(y_train, dt.predict(X_train))
    acc_te = accuracy_score(y_test,  dt.predict(X_test))
    label = str(prof) if prof is not None else 'Sin límite'
    resultados.append({'depth': label, 'train': acc_tr, 'test': acc_te})
    print(f'depth={label:8s}  train={acc_tr:.4f}  test={acc_te:.4f}')

In [ ]:
# --- Random Forest con OOB score ------------------------------------
bosque = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features='sqrt',
    oob_score=True,       # activa la estimación out-of-bag
    random_state=semilla,
    n_jobs=-1,
)
bosque.fit(X_train, y_train)

acc_rf_train = accuracy_score(y_train, bosque.predict(X_train))
acc_rf_test  = accuracy_score(y_test,  bosque.predict(X_test))

print(f'Train:     {acc_rf_train:.4f}')
print(f'OOB score: {bosque.oob_score_:.4f}')  # estimación real
print(f'Test:      {acc_rf_test:.4f}')

In [ ]:
# --- Importancia de features ----------------------------------------
importancias = pd.Series(
    bosque.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importancias.round(4))
# cp         0.1423   <- tipo de dolor torácico
# thalach    0.1318   <- frecuencia cardíaca máxima
# oldpeak    0.1204   <- depresión del ST
# ca         0.1089   <- vasos coloreados por fluoroscopía
# ...

# --- Gráfico de barras -----------------------------------------------
importancias.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.xlabel('Importancia media (Gini)')
plt.title('Importancia de features — Random Forest Heart Disease')
plt.gca().invert_yaxis()   # la más importante arriba
plt.tight_layout()
plt.show()

In [ ]:
# --- Informe detallado de clasificación: RF -------------------------
print(classification_report(
    y_test,
    bosque.predict(X_test),
    target_names=['Sin enfermedad', 'Con enfermedad'],
))

In [ ]:
# --- Pipeline con Random Forest --------------------------------------
pipeline_rf = Pipeline(steps=[
    ('imputar', SimpleImputer(strategy='median')),
    # StandardScaler omitido: RF no requiere escalado
    ('modelo', RandomForestClassifier(
        n_estimators=200,
        random_state=semilla,
        n_jobs=-1,
    )),
])

pipeline_rf.fit(X_train, y_train)
print(f'Pipeline RF — Test: {accuracy_score(y_test, pipeline_rf.predict(X_test)):.4f}')

# Validación cruzada sobre el pipeline
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring='accuracy')
print(f'CV Media: {scores.mean():.4f} ± {scores.std():.4f}')